# 06 - Modelagem ARIMA para Previsao

## Objetivo
Construir e validar modelos ARIMA para previsao de IPCA.

## Fluxo
1. Carregar dados
2. Testes de estacionariedade (ADF)
3. Determinacao de parametros ARIMA
4. Ajuste do modelo
5. Validacao e previsoes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)

## 1. Carregar dados

In [ ]:
caminho_dados = Path('../dados/brutos/indicadores_consolidados.csv')
df = pd.read_csv(caminho_dados)
df['data'] = pd.to_datetime(df['data'])
df = df.sort_values('data').reset_index(drop=True)

# Serie de IPCA mensal
ipca = df['ipca_mensal']

print(f'Serie IPCA Mensal carregada: {len(ipca)} observacoes')

## 2. Teste de Estacionariedade (ADF)

In [ ]:
# Teste ADF na serie original
adf_result = adfuller(ipca, autolag='AIC')

print('TESTE DE RAIZ UNITARIA (ADF):')
print('=' * 50)
print(f'Estatistica de teste: {adf_result[0]:.6f}')
print(f'P-valor: {adf_result[1]:.6f}')
print(f'Lags usados: {adf_result[2]}')
print()

if adf_result[1] < 0.05:
    print('Resultado: SERIE ESTACIONARIA (rejeitar H0)')
    print('Conclusao: Pode usar ARIMA com d=0')
    d = 0
else:
    print('Resultado: SERIE NAO-ESTACIONARIA (falhar em rejeitar H0)')
    print('Conclusao: Precisa diferenciar (d>=1)')
    d = 1

print()

# Teste na primeira diferenca
ipca_diff = ipca.diff().dropna()
adf_diff = adfuller(ipca_diff, autolag='AIC')

print('TESTE ADF NA PRIMEIRA DIFERENCA:')
print(f'P-valor: {adf_diff[1]:.6f}')
if adf_diff[1] < 0.05:
    print('Serie diferenciada eh estacionaria')

## 3. ACF e PACF para determinacao de p e q

In [ ]:
# Calcular ACF e PACF
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ACF
acf_vals = acf(ipca_diff, nlags=20)
ax1.vlines(range(len(acf_vals)), 0, acf_vals, color='steelblue', alpha=0.8)
ax1.axhline(y=0, color='black', linewidth=1)
ax1.fill_between(range(len(acf_vals)), -1.96/np.sqrt(len(ipca_diff)), 1.96/np.sqrt(len(ipca_diff)), alpha=0.2)
ax1.set_title('Funcao de Autocorrelacao (ACF)', fontsize=12, fontweight='bold')
ax1.set_ylabel('ACF')
ax1.set_xlabel('Lag')
ax1.grid(True, alpha=0.3)

# PACF
pacf_vals = pacf(ipca_diff, nlags=20)
ax2.vlines(range(len(pacf_vals)), 0, pacf_vals, color='coral', alpha=0.8)
ax2.axhline(y=0, color='black', linewidth=1)
ax2.fill_between(range(len(pacf_vals)), -1.96/np.sqrt(len(ipca_diff)), 1.96/np.sqrt(len(ipca_diff)), alpha=0.2)
ax2.set_title('Funcao de Autocorrelacao Parcial (PACF)', fontsize=12, fontweight='bold')
ax2.set_ylabel('PACF')
ax2.set_xlabel('Lag')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('ACF e PACF calculados')
print('Usados para determinar p (PACF) e q (ACF)')

## 4. Ajuste de modelo ARIMA

In [ ]:
# Dividir em treino e teste (80-20)
tamanho_treino = int(len(ipca) * 0.8)
treino = ipca[:tamanho_treino]
teste = ipca[tamanho_treino:]

print(f'Tamanho treino: {len(treino)}')
print(f'Tamanho teste: {len(teste)}')
print()

# Ajustar ARIMA(1,1,1)
modelo = ARIMA(treino, order=(1, 1, 1))
resultado = modelo.fit()

print('MODELO ARIMA(1,1,1) AJUSTADO')
print(resultado.summary())

## 5. Previsoes no conjunto de teste

In [ ]:
# Fazer previsoes
previsoes = resultado.get_forecast(steps=len(teste))
previsoes_df = previsoes.summary_frame()

# Metricas de erro
rmse = np.sqrt(mean_squared_error(teste, previsoes_df['mean']))
mae = mean_absolute_error(teste, previsoes_df['mean'])
mape = np.mean(np.abs((teste - previsoes_df['mean']) / teste * 100))

print('METRICAS DE ERRO (CONJUNTO DE TESTE):')
print('=' * 50)
print(f'RMSE: {rmse:.4f}%')
print(f'MAE: {mae:.4f}%')
print(f'MAPE: {mape:.2f}%')

## 6. Visualizacao das previsoes

In [ ]:
plt.figure(figsize=(14, 7))

# Dados historicos
plt.plot(range(len(treino)), treino, label='Treino', linewidth=2, color='blue')
plt.plot(range(len(treino), len(ipca)), teste, label='Teste (Real)', linewidth=2, color='green')

# Previsoes
plt.plot(range(len(treino), len(ipca)), previsoes_df['mean'], 
         label='Previsao ARIMA', linewidth=2, color='red', linestyle='--')

# Intervalo de confianca
plt.fill_between(range(len(treino), len(ipca)),
                  previsoes_df['mean_ci_lower'],
                  previsoes_df['mean_ci_upper'],
                  alpha=0.2, color='red', label='IC 95%')

plt.title('IPCA Mensal - Modelo ARIMA(1,1,1)', fontsize=14, fontweight='bold')
plt.xlabel('Periodo')
plt.ylabel('IPCA (%)')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Previsao para proximo periodo

In [ ]:
# Retreinar no dataset completo
modelo_final = ARIMA(ipca, order=(1, 1, 1))
resultado_final = modelo_final.fit()

# Previsao para os proximos 6 meses
previsao_futura = resultado_final.get_forecast(steps=6)
previsao_df = previsao_futura.summary_frame()

print('PREVISAO PARA PROXIMOS 6 MESES:')
print('=' * 70)
print('Mes | Previsto | IC Inferior | IC Superior')
print('-' * 70)
for i in range(len(previsao_df)):
    print(f' {i+1:2} | {previsao_df.iloc[i]["mean"]:8.3f}% | {previsao_df.iloc[i]["mean_ci_lower"]:10.3f}% | {previsao_df.iloc[i]["mean_ci_upper"]:11.3f}%')

## 8. Resumo da modelagem

In [ ]:
print('
RESUMO DA MODELAGEM ARIMA:')
print('=' * 70)
print()
print('MODELO: ARIMA(1,1,1)')
print('  p=1: Componente autoregressivo')
print('  d=1: Uma diferenciacao')
print('  q=1: Componente media movel')
print()
print('DESEMPENHO NO TESTE:')
print(f'  RMSE: {rmse:.4f}%')
print(f'  MAE: {mae:.4f}%')
print(f'  MAPE: {mape:.2f}%')
print()
print('PROXIMAS PREVISOES:')
print(f'  Proxima previsao (1 mes): {previsao_df.iloc[0]["mean"]:.3f}%')
print(f'  Intervalo de confianca (95%): [{previsao_df.iloc[0]["mean_ci_lower"]:.3f}%, {previsao_df.iloc[0]["mean_ci_upper"]:.3f}%]')